<a href="https://colab.research.google.com/github/YomnaEsmail/Masters/blob/main/%F0%9F%91%80Predicting_5minutes_7Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 👀About this Code
The important change is that this is no longer a one-step prediction problem (PL=1). All six models should now use:

LB = 24 historical time steps as paper

PL = 1 future time steps as paper [5 minutes]

Batch size = 128 as paper

Epochs = 120 as paper

5 independent runs

65:35 chronological train/test split

8 feature-selection methods

Save every run's predictions, metrics, training history, model weights, and configuration

Generate graphs for each model and overall comparison

# Adding M7 which is my own pCNN-BiLSTM Structure

**M1: Standard LSTM**

* **Description:** A sequential recurrent architecture with two stacked LSTM layers to extract temporal dependencies, followed by a dense decision layer.
* **Parameters & Specs:**
* **Input Layer:** Shape `(lookback, features)`
* **Recurrent Layers:** First LSTM (128 units, `return_sequences=True`), second LSTM (64 units)
* **Regularization:** Dropout ($0.2$) after each LSTM layer
* **Dense Layers:** Dense (32 units, ReLU activation), Output Dense (`PL` units, linear activation)
* **Compilation:** Adam optimizer, MSE loss, MAE metric



**M2: Bidirectional LSTM (BiLSTM)**

* **Description:** Evaluates time series data in both forward and backward temporal directions using stacked bidirectional wrappers.
* **Parameters & Specs:**
* **Input Layer:** Shape `(lookback, features)`
* **Recurrent Layers:** First BiLSTM (64 units per direction, `return_sequences=True`), second BiLSTM (32 units per direction)
* **Regularization:** Dropout ($0.2$) after each BiLSTM layer
* **Dense Layers:** Dense (32 units, ReLU activation), Output Dense (`PL` units, linear activation)
* **Compilation:** Adam optimizer, MSE loss, MAE metric



**M3: Standard CNN-LSTM**

* **Description:** Combines 1D convolutional layers to extract local spatial/temporal feature maps with an LSTM layer for sequence dynamics.
* **Parameters & Specs:**
* **Input Layer:** Shape `(lookback, features)`
* **Convolutional Blocks:** Two 1D Conv layers (64 filters each, kernel size 4, ReLU activation), MaxPooling1D (pool size 2)
* **Regularization:** Dropout ($0.2$) post-pooling
* **Recurrent & Dense Layers:** LSTM (100 units), Dense (50 units, ReLU activation), Output Dense (`PL` units, linear activation)
* **Compilation:** Adam optimizer, MSE loss, MAE metric



**M4: Standard CNN-BiLSTM**

* **Description:** Integrates a single 1D convolutional feature extractor with stacked bidirectional LSTMs to capture complex patterns across the entire sequence.
* **Parameters & Specs:**
* **Input Layer:** Shape `(lookback, features)`
* **Convolutional Block:** 1D Conv layer (64 filters, kernel size 4, ReLU activation), MaxPooling1D (pool size 2)
* **Regularization:** Dropout ($0.2$) post-pooling, Dropout ($0.2$) between BiLSTMs
* **Recurrent Layers:** First BiLSTM (64 units, `return_sequences=True`), second BiLSTM (32 units)
* **Dense Layers:** Dense (64 units, ReLU activation), Output Dense (`PL` units, linear activation)
* **Compilation:** Adam optimizer, MSE loss, MAE metric



**M5: Parallel CNN-LSTM (pCNN-LSTM)**

* **Description:** A custom functional architecture running three parallel 1D convolutional branches with varying dilation rates to capture multi-scale temporal receptive fields before passing features to an LSTM.
* **Parameters & Specs:**
* **Input Layer:** Shape `(lookback, features)`
* **Parallel Conv Branches:** 3 branches (64 filters each, kernel size 4, causal padding, max-norm constraint of 5.0). Branch 1: dilation rate 1; Branch 2: dilation rate 2; Branch 3: dilation rate 4
* **Merging:** Concatenation layer joining all 3 parallel branches
* **Recurrent & Dense Layers:** LSTM (100 units, internal dropout $0.1$, max-norm constraint 5.0), Dense (50 units, ReLU activation), Output Dense (`PL` units, linear activation)
* **Compilation:** Adam optimizer, MSE loss, MAE metric



**M6: Parallel CNN-BiLSTM (pCNN-BiLSTM)**

* **Description:** An advanced multi-branch architecture using stacked dilated convolutions to extract temporal patterns across different temporal depths, fed into a heavy Bidirectional LSTM layer.
* **Parameters & Specs:**
* **Input Layer:** Shape `(lookback, features)`
* **Branch 1 (Stacked):** Conv1D (64 filters, kernel size 3, dilation 1) $\rightarrow$ Conv1D (64 filters, kernel size 3, dilation 2)
* **Branch 2:** *(Commented out/Disabled)*
* **Branch 3 (Single-layer):** Conv1D (64 filters, kernel size 3, dilation 1)
* **Merging:** Concatenation layer combining output of Branch 1 and Branch 3
* **Recurrent & Output Layers:** BiLSTM (100 units, max-norm constraint 5.0), Output Dense (`pred_length` units, linear activation)
* **Compilation:** Adam optimizer (learning rate $0.05$, gradient norm clipping $5.0$), MSE loss, MAE metric

In [1]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU Successfully Activated: {gpus[0].name}")
else:
    print("GPU not detected. Make sure Runtime settings are saved.")

GPU Successfully Activated: /physical_device:GPU:0


In [2]:
# Prevent TensorFlow from pre-allocating all VRAM at once
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [3]:
# =========================================================
# 1. IMPORTS
# =========================================================

import os
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import time
import random
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from tensorflow.keras.callbacks import LearningRateScheduler
from matplotlib.backends.backend_pdf import PdfPages

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from sklearn.feature_selection import (
    mutual_info_regression,
    RFE,
    SelectKBest,
    f_regression
)

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    LSTM,
    GRU,
    Conv1D,
    Dropout,
    Bidirectional,
    Input
)

from tensorflow.keras.constraints import max_norm
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
import numpy as np
import pandas as pd
import os
import time

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    LSTM,
    Bidirectional,
    GRU,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Conv1D, Concatenate, LSTM, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.layers import Input, Conv1D, Concatenate, LSTM, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Bidirectional, LSTM, Dense, Dropout
warnings.filterwarnings("ignore")



**Loading Data & Feature Selection Preparation**

In [5]:
# =========================================================
# 2. REPRODUCIBILITY
# =========================================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# =========================================================
# 3. STYLE
# =========================================================

sns.set_style("whitegrid")
sns.set_context("talk")

palette = sns.color_palette("Set2")

# =========================================================
# 4. LOAD DATA
# =========================================================

DATA_PATH = "43_cleaned_original.csv"
TARGET = "CPUusageMHZ"

df = pd.read_csv(DATA_PATH)

print("\nDataset Shape:", df.shape)

if TARGET not in df.columns:
    raise KeyError(f"Target column '{TARGET}' not found.")

print("\nColumns:")
print(df.columns.tolist())

# =========================================================
# 5. FEATURE SELECTION
# =========================================================

def generate_feature_sets(df, target_col):

    cols = df.columns.tolist()

    drop_cols = [target_col]

    if "Timestampms" in cols:
        drop_cols.append("Timestampms")

    feature_cols = df.columns.drop(drop_cols)

    X = df[feature_cols]
    y = df[target_col]

    X_scaled = StandardScaler().fit_transform(X)

    fs = {}

    # -----------------------------------------------------
    # 1. Correlation
    # -----------------------------------------------------

    fs["Correlation"] = (
        X.corrwith(y)
        .abs()
        .sort_values(ascending=False)
        .head(5)
        .index
        .tolist()
    )

    # -----------------------------------------------------
    # 2. Mutual Information
    # -----------------------------------------------------

    mi = mutual_info_regression(X_scaled, y)

    fs["MutualInfo"] = (
        pd.Series(mi, index=X.columns)
        .sort_values(ascending=False)
        .head(5)
        .index
        .tolist()
    )

    # -----------------------------------------------------
    # 3. Random Forest
    # -----------------------------------------------------

    rf = RandomForestRegressor(
        n_estimators=100,
        random_state=SEED
    )

    rf.fit(X, y)

    fs["RandomForest"] = (
        pd.Series(rf.feature_importances_, index=X.columns)
        .sort_values(ascending=False)
        .head(5)
        .index
        .tolist()
    )

    # -----------------------------------------------------
    # 4. RFE
    # -----------------------------------------------------

    rfe = RFE(
        RandomForestRegressor(
            n_estimators=50,
            random_state=SEED
        ),
        n_features_to_select=5
    )

    rfe.fit(X, y)

    fs["RFE"] = X.columns[rfe.support_].tolist()

    # -----------------------------------------------------
    # 5. Lasso
    # -----------------------------------------------------

    lasso = LassoCV(random_state=SEED)

    lasso.fit(X_scaled, y)

    fs["Lasso"] = (
        pd.Series(np.abs(lasso.coef_), index=X.columns)
        .sort_values(ascending=False)
        .head(5)
        .index
        .tolist()
    )

    # -----------------------------------------------------
    # 6. PCA
    # -----------------------------------------------------

    pca = PCA(n_components=5)

    pca.fit(X_scaled)

    fs["PCA"] = (
        X.columns[
            np.argsort(np.abs(pca.components_[0]))[::-1][:5]
        ].tolist()
    )

    # -----------------------------------------------------
    # 7. Permutation Importance
    # -----------------------------------------------------

    perm = permutation_importance(
        rf,
        X,
        y,
        n_repeats=10,
        random_state=SEED
    )

    fs["Permutation"] = (
        X.columns[
            np.argsort(perm.importances_mean)[::-1][:5]
        ].tolist()
    )

    # -----------------------------------------------------
    # 8. SelectKBest
    # -----------------------------------------------------

    skb = SelectKBest(f_regression, k=5)

    skb.fit(X_scaled, y)

    fs["SelectKBest"] = (
        X.columns[skb.get_support()]
        .tolist()
    )

    return fs


feature_sets = generate_feature_sets(df, TARGET)

print("\nGenerated Feature Sets:\n")

for k, v in feature_sets.items():
    print(f"{k}: {v}")



Dataset Shape: (8632, 11)

Columns:
['Timestampms', 'CPUcores', 'CPUcapacityprovisionedMHZ', 'CPUusageMHZ', 'CPUusage%', 'MemorycapacityprovisionedKB', 'MemoryusageKB', 'DiskreadthroughputKBs', 'DiskwritethroughputKBs', 'NetworkreceivedthroughputKBs', 'NetworktransmittedthroughputKBs']

Generated Feature Sets:

Correlation: ['CPUusage%', 'DiskreadthroughputKBs', 'MemoryusageKB', 'DiskwritethroughputKBs', 'NetworktransmittedthroughputKBs']
MutualInfo: ['CPUusage%', 'NetworktransmittedthroughputKBs', 'NetworkreceivedthroughputKBs', 'DiskwritethroughputKBs', 'MemoryusageKB']
RandomForest: ['CPUusage%', 'CPUcapacityprovisionedMHZ', 'NetworktransmittedthroughputKBs', 'MemoryusageKB', 'NetworkreceivedthroughputKBs']
RFE: ['CPUcapacityprovisionedMHZ', 'CPUusage%', 'MemoryusageKB', 'DiskreadthroughputKBs', 'NetworktransmittedthroughputKBs']
Lasso: ['CPUusage%', 'CPUcores', 'CPUcapacityprovisionedMHZ', 'MemorycapacityprovisionedKB', 'MemoryusageKB']
PCA: ['DiskwritethroughputKBs', 'Networkrece

In [6]:
# =========================================================
# MULTI-STEP FORECASTING CONFIGURATION
# =========================================================

LB = 24                  # Lookback window
PL = 1                  # Prediction horizon
BATCH_SIZE = 128
EPOCHS = 120
N_RUNS = 5

TRAIN_RATIO = 0.65
TEST_RATIO = 1 - TRAIN_RATIO
print("\n======================================================")
print("MULTI-STEP FORECASTING CONFIGURATION")
print("======================================================")
print(f"Lookback Window (LB): {LB}")
print(f"Prediction Length (PL): {PL}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Runs: {N_RUNS}")
print(f"Train/Test Split: {TRAIN_RATIO:.0%}/{1-TRAIN_RATIO:.0%}")


MULTI-STEP FORECASTING CONFIGURATION
Lookback Window (LB): 24
Prediction Length (PL): 12
Batch Size: 128
Epochs: 120
Runs: 5
Train/Test Split: 65%/35%


In [7]:
# =========================================================
# MULTI-STEP SEQUENCE CREATION
# =========================================================

def create_multistep_sequences(
    X,
    y,
    lookback,
    pred_length
):

    X_seq = []
    y_seq = []

    for i in range(
        lookback,
        len(X) - pred_length + 1
    ):

        X_seq.append(
            X[i-lookback:i]
        )

        y_seq.append(
            y[i:i+pred_length]
        )

    return (
        np.array(X_seq),
        np.array(y_seq)
    )

In [9]:
# =========================================================
# M1: LSTM - MULTI-STEP
# =========================================================

def build_M1_LSTM(input_shape):

    model = Sequential([

        Input(shape=input_shape),

        LSTM(
            128,
            return_sequences=True
        ),

        Dropout(0.2),

        LSTM(64),

        Dropout(0.2),

        Dense(
            32,
            activation="relu"
        ),

        Dense(
            PL,
            activation="linear"
        )
    ])

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    return model
# =========================================================
# M2: BiLSTM - MULTI-STEP
# =========================================================

def build_M2_BiLSTM(input_shape):

    model = Sequential([

        Input(shape=input_shape),

        Bidirectional(
            LSTM(
                64,
                return_sequences=True
            )
        ),

        Dropout(0.2),

        Bidirectional(
            LSTM(32)
        ),

        Dropout(0.2),

        Dense(
            32,
            activation="relu"
        ),

        Dense(
            PL,
            activation="linear"
        )
    ])

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    return model

# =========================================================
# M3: CNN-LSTM - MULTI-STEP
# =========================================================

def build_M3_CNNLSTM(input_shape):

    model = Sequential([

        Input(shape=input_shape),

        Conv1D(
            filters=64,
            kernel_size=4,
            activation="relu"
        ),

        Conv1D(
            filters=64,
            kernel_size=4,
            activation="relu"
        ),

        MaxPooling1D(
            pool_size=2
        ),

        Dropout(0.2),

        LSTM(100),

        Dense(
            50,
            activation="relu"
        ),

        Dense(
            PL,
            activation="linear"
        )
    ])

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    return model

# =========================================================
# M4: CNN-BiLSTM - MULTI-STEP
# =========================================================

def build_M4_CNNBiLSTM(input_shape):

    model = Sequential([

        Input(shape=input_shape),

        Conv1D(
            filters=64,
            kernel_size=4,
            activation="relu"
        ),

        MaxPooling1D(
            pool_size=2
        ),

        Dropout(0.2),

        Bidirectional(
            LSTM(
                64,
                return_sequences=True
            )
        ),

        Dropout(0.2),

        Bidirectional(
            LSTM(32)
        ),

        Dense(
            64,
            activation="relu"
        ),

        Dense(
            PL,
            activation="linear"
        )
    ])

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    return model

# =========================================================
# M5: pCNN-LSTM - MULTI-STEP my own structure
# =========================================================

def build_M5_pCNNLSTM(input_shape):

    input_seq = Input(
        shape=input_shape
    )

    # Branch 1
    conv1 = Conv1D(filters=64,kernel_size=4,dilation_rate=1,padding="causal",activation="relu",kernel_constraint=max_norm(5))(input_seq)

    # Branch 2
    conv2 = Conv1D(filters=64,kernel_size=4,dilation_rate=2,padding="causal",activation="relu",kernel_constraint=max_norm(5))(input_seq)

    # Branch 3
    conv3 = Conv1D(filters=64,kernel_size=4,dilation_rate=4,padding="causal",activation="relu",kernel_constraint=max_norm(5))(input_seq)

    # Merge
    merged = Concatenate()([
        conv1,
        conv2,
        conv3
    ])

    lstm_out = LSTM(
        100,
        dropout=0.1,
        kernel_constraint=max_norm(5)
    )(merged)

    dense1 = Dense(
        50,
        activation="relu"
    )(lstm_out)

    output = Dense(
        PL,
        activation="linear"
    )(dense1)

    model = Model(
        inputs=input_seq,
        outputs=output
    )

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    return model

# =========================================================
# M6: pCNN-BiLSTM - PAPER ARCHITECTURE same as pCNN-LSTM  CB1 & CB2 two layers
# LB=24 / PL=1 will follow same structure as the paper but BiLSTM 100
# =========================================================

def build_M6_pCNNBiLSTM(
    input_shape,
    pred_length=1
):

    input_seq = Input(
        shape=input_shape
    )

    # -----------------------------------------------------
    # Branch 1
    # KS=3, DL=1
    # followed by KS=3, DL=2
    # -----------------------------------------------------

    cb1_l1 = Conv1D(filters=64,kernel_size=3,dilation_rate=1,padding="causal",activation="relu",kernel_constraint=max_norm(5.0))(input_seq)

    cb1_l2 = Conv1D(filters=64,kernel_size=3,dilation_rate=2,padding="causal",activation="relu",kernel_constraint=max_norm(5.0))(cb1_l1)

    # -----------------------------------------------------
    # Branch 2
    # KS=3, DL=1
    # followed by KS=6, DL=2
    # -----------------------------------------------------

    #cb2_l1 = Conv1D(filters=64,kernel_size=3,dilation_rate=1,padding="causal",activation="relu",kernel_constraint=max_norm(5.0))(input_seq)

   # cb2_l2 = Conv1D(filters=64,kernel_size=6,dilation_rate=2,padding="causal",activation="relu",kernel_constraint=max_norm(5.0))(cb2_l1)

    # -----------------------------------------------------
    # Branch 3
    # KS=3, DL=1
    # -----------------------------------------------------

    cb3_l1 = Conv1D(filters=64,kernel_size=3,dilation_rate=1,padding="causal",activation="relu",kernel_constraint=max_norm(5.0))(input_seq)

    # -----------------------------------------------------
    # Merge CNN branches
    # -----------------------------------------------------

    merged = Concatenate()([
        cb1_l2,
        cb3_l1
    ]) #cb2_l2, removed

    # -----------------------------------------------------
    # BiLSTM
    # -----------------------------------------------------

    bilstm_out = Bidirectional(
        LSTM(
            100,
            kernel_constraint=max_norm(5.0)
        )
    )(merged)

    # -----------------------------------------------------
    # Multi-step output
    # -----------------------------------------------------

    output = Dense(
        pred_length,
        activation="linear"
    )(bilstm_out)

    model = Model(
        inputs=input_seq,
        outputs=output
    )

    # -----------------------------------------------------
    # Optimizer
    # -----------------------------------------------------

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=0.05,
        clipnorm=5.0
    )

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )

    return model

# ---------------------------------------------------------
# M7: pCNN-BiLSTM - same parameters as pCNN-LSTM my parameters
# ---------------------------------------------------------

def build_M7_pCNNBiLSTM(input_shape,pred_length=12):

    input_seq = Input(shape=input_shape)

    # Branch 1: short-range patterns
    conv1 = Conv1D(filters=64, kernel_size=4, dilation_rate=1, padding='causal',
                   activation='relu', kernel_constraint=max_norm(5))(input_seq)

    # Branch 2: medium-range patterns
    conv2 = Conv1D(filters=64, kernel_size=4, dilation_rate=2, padding='causal',
                   activation='relu', kernel_constraint=max_norm(5))(input_seq)

    # Branch 3: long-range patterns
    conv3 = Conv1D(filters=64, kernel_size=4, dilation_rate=4, padding='causal',
                   activation='relu', kernel_constraint=max_norm(5))(input_seq)

    # Combine the three parallel feature branches
    merged = Concatenate()([conv1, conv2, conv3])

    # --- MODIFICATION: Wrap LSTM with Bidirectional ---
    # Optional: reduce units to 50 if you want to keep the total output dim (50 forward + 50 backward = 100)
    bilstm_out = Bidirectional(LSTM(50, dropout=0.1, kernel_constraint=max_norm(5)))(merged)

    dense1 = Dense(50, activation='relu')(bilstm_out)
    #output = Dense(1)(dense1) # for PL=1
    output = Dense(pred_length,activation="linear")(dense1) #for PL=6 | PL=12

    model = Model(
        inputs=input_seq,
        outputs=output
    )

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    return model

In [10]:
# =========================================================
# LEARNING RATE DECAY same as pCNN-LSTM Paper parameters
# =========================================================

def step_decay(epoch):

    initial_lr = 0.05
    drop_rate = 0.5
    epochs_drop = 30

    return float(
        initial_lr *
        (
            drop_rate **
            (epoch // epochs_drop)
        )
    )

In [11]:
# =========================================================
# MODEL LIST
# =========================================================

models = [

    (
        "Model 1: LSTM",
        build_M1_LSTM,
        "standard"
    ),

    (
        "Model 2: BiLSTM",
        build_M2_BiLSTM,
        "standard"
    ),

    (
        "Model 3: CNN-LSTM",
        build_M3_CNNLSTM,
        "standard"
    ),

    (
        "Model 4: CNN-BiLSTM",
        build_M4_CNNBiLSTM,
        "standard"
    ),

    (
        "Model 5: pCNN-LSTM",
        build_M5_pCNNLSTM,
        "standard"
    ),

    (
        "Model 6: pCNN-BiLSTM",
        build_M6_pCNNBiLSTM,
        "paper"
    ),

    (
        "Model 7: pCNN_BiLSTM",
        build_M7_pCNNBiLSTM,
        "paper"
    )

]

In [ ]:
Raw data
   ↓
65% Train | 35% Test
   ↓
Fit scaler ONLY on Train
   ↓
Transform Train + Test using Train scaler
   ↓
Create LB=48 sequences
   ↓
Predict PL=12

SyntaxError: invalid character '↓' (U+2193) (3131988652.py, line 2)

In [12]:
# =========================================================
# GOOGLE DRIVE
# =========================================================

import os
import json
import time
import random
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from google.colab import drive

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from tensorflow.keras.callbacks import LearningRateScheduler


# =========================================================
# MOUNT GOOGLE DRIVE
# =========================================================

drive.mount('/content/drive')


# =========================================================
# BASE DIRECTORY
# =========================================================
# Everything will be saved under:
# Google Drive > MyDrive > Time_Series_Experiments > PL12_Results

#BASE_DIR = "/content/drive/MyDrive/Time_Series_Experiments/PL12_Results"

# Google Drive root for this project
PROJECT_DIR = "/content/drive/MyDrive/Time_Series_Experiments"

# Results for this particular experiment/model
BASE_DIR = os.path.join(PROJECT_DIR, "PL12_ResultsM7")
# =========================================================
# OUTPUT DIRECTORIES
# =========================================================

DIR_PREDICTIONS = os.path.join(
    BASE_DIR,
    "Predictions"
)

DIR_MODELS = os.path.join(
    BASE_DIR,
    "Saved_Models"
)

DIR_HISTORY = os.path.join(
    BASE_DIR,
    "Training_History"
)

DIR_METRICS = os.path.join(
    BASE_DIR,
    "Metrics"
)

DIR_GRAPHS = os.path.join(
    BASE_DIR,
    "Graphs"
)

DIR_PER_HORIZON = os.path.join(
    BASE_DIR,
    "Per_Horizon_Metrics"
)

DIR_SCALERS = os.path.join(
    BASE_DIR,
    "Scalers"
)

DIR_SEQUENCES = os.path.join(
    BASE_DIR,
    "Sequences"
)

DIR_CONFIG = os.path.join(
    BASE_DIR,
    "Configurations"
)

DIR_SUMMARY = os.path.join(
    BASE_DIR,
    "Model_Summaries"
)


# =========================================================
# CREATE DIRECTORIES
# =========================================================

ALL_DIRS = [
    BASE_DIR,
    DIR_PREDICTIONS,
    DIR_MODELS,
    DIR_HISTORY,
    DIR_METRICS,
    DIR_GRAPHS,
    DIR_PER_HORIZON,
    DIR_SCALERS,
    DIR_SEQUENCES,
    DIR_CONFIG,
    DIR_SUMMARY
]

for directory in ALL_DIRS:
    os.makedirs(directory, exist_ok=True)


# =========================================================
# VERIFY
# =========================================================

print("All output directories are ready.")
print()
print("Base directory:")
print(BASE_DIR)
print()

for directory in ALL_DIRS:
    print(directory)

Mounted at /content/drive
All output directories are ready.

Base directory:
/content/drive/MyDrive/Time_Series_Experiments/PL12_ResultsM7

/content/drive/MyDrive/Time_Series_Experiments/PL12_ResultsM7
/content/drive/MyDrive/Time_Series_Experiments/PL12_ResultsM7/Predictions
/content/drive/MyDrive/Time_Series_Experiments/PL12_ResultsM7/Saved_Models
/content/drive/MyDrive/Time_Series_Experiments/PL12_ResultsM7/Training_History
/content/drive/MyDrive/Time_Series_Experiments/PL12_ResultsM7/Metrics
/content/drive/MyDrive/Time_Series_Experiments/PL12_ResultsM7/Graphs
/content/drive/MyDrive/Time_Series_Experiments/PL12_ResultsM7/Per_Horizon_Metrics
/content/drive/MyDrive/Time_Series_Experiments/PL12_ResultsM7/Scalers
/content/drive/MyDrive/Time_Series_Experiments/PL12_ResultsM7/Sequences
/content/drive/MyDrive/Time_Series_Experiments/PL12_ResultsM7/Configurations
/content/drive/MyDrive/Time_Series_Experiments/PL12_ResultsM7/Model_Summaries


In [13]:
# =========================================================
# SAFE FILE NAME
# =========================================================

def make_safe_name(text):

    return (
        str(text)
        .replace(":", "")
        .replace(" ", "_")
        .replace("-", "")
        .replace("/", "_")
        .replace("\\", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("[", "")
        .replace("]", "")
        .replace("%", "pct")
    )

In [14]:
# =========================================================
# MAPE FUNCTION
# =========================================================

def calculate_mape(
    y_true,
    y_pred
):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    non_zero = y_true != 0

    if not np.any(non_zero):
        return np.nan

    return (
        np.mean(
            np.abs(
                (
                    y_true[non_zero]
                    -
                    y_pred[non_zero]
                )
                /
                y_true[non_zero]
            )
        )
        * 100
    )

In [15]:
# =========================================================
# SAVE MODEL SUMMARY
# =========================================================

def save_model_summary(
    model,
    path
):

    with open(
        path,
        "w"
    ) as f:

        model.summary(
            print_fn=lambda x: f.write(
                x + "\n"
            )
        )

In [16]:
# =========================================================
# SAVE TRAINING LOSS GRAPH
# =========================================================

def save_training_graph(
    history,
    prefix
):

    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        history.history["loss"],
        label="Training Loss"
    )

    if "val_loss" in history.history:

        plt.plot(
            history.history["val_loss"],
            label="Validation Loss"
        )

    plt.xlabel(
        "Epoch"
    )

    plt.ylabel(
        "Loss"
    )

    plt.title(
        f"Training and Validation Loss - {prefix}"
    )

    plt.legend()

    plt.tight_layout()

    path = os.path.join(
        DIR_GRAPHS,
        prefix + "_TrainingLoss.png"
    )

    plt.savefig(
        path,
        dpi=300
    )

    plt.close()

    return path

In [17]:
# =========================================================
# SAVE ACTUAL VS PREDICTED GRAPH
# =========================================================

def save_prediction_graphs(
    y_true,
    y_pred,
    prefix
):

    graph_paths = []

    horizons = [
        0,
        y_true.shape[1] - 1
    ]

    for h in horizons:

        plt.figure(
            figsize=(12, 5)
        )

        plt.plot(
            y_true[:, h],
            label=f"Actual t+{h+1}"
        )

        plt.plot(
            y_pred[:, h],
            label=f"Predicted t+{h+1}"
        )

        plt.xlabel(
            "Test Sequence"
        )

        plt.ylabel(
            TARGET
        )

        plt.title(
            f"Actual vs Predicted - "
            f"t+{h+1} - {prefix}"
        )

        plt.legend()

        plt.tight_layout()

        path = os.path.join(
            DIR_GRAPHS,
            prefix +
            f"_Actual_vs_Predicted_t+{h+1}.png"
        )

        plt.savefig(
            path,
            dpi=300
        )

        plt.close()

        graph_paths.append(
            path
        )

    return graph_paths

In [18]:
# =========================================================
# STORAGE
# =========================================================

results = []
horizon_results = []

predictions = {}
histories = {}

In [19]:
# =========================================================
# EXISTING RESULTS
# =========================================================

MAIN_RESULTS_FILE = os.path.join(
    DIR_METRICS,
    "ALL_RUN_RESULTS_PL12M7.csv"
)

HORIZON_RESULTS_FILE = os.path.join(
    DIR_PER_HORIZON,
    "ALL_HORIZON_RESULTS_PL12M7.csv"
)


# =========================================================
# LOAD PREVIOUS RESULTS IF THEY EXIST
# =========================================================

if os.path.exists(
    MAIN_RESULTS_FILE
):

    existing_results = pd.read_csv(
        MAIN_RESULTS_FILE
    )

    results = existing_results.to_dict(
        orient="records"
    )

    print(
        "Loaded existing main results:",
        len(results)
    )


if os.path.exists(
    HORIZON_RESULTS_FILE
):

    existing_horizon_results = pd.read_csv(
        HORIZON_RESULTS_FILE
    )

    horizon_results = (
        existing_horizon_results
        .to_dict(
            orient="records"
        )
    )

    print(
        "Loaded existing horizon results:",
        len(horizon_results)
    )


# =========================================================
# COMPLETED RUN IDENTIFIERS
# =========================================================

completed_runs = set()

for row in results:

    completed_runs.add(
        (
            row["Model"],
            row["Feature Selection"],
            int(row["Run"])
        )
    )


print(
    "Previously completed runs:",
    len(completed_runs)
)


# =========================================================
# MAIN EXPERIMENT
# =========================================================
START_MODEL_INDEX = 6   # Model 6 = index 5 [START_MODEL_INDEX:]    M7--index 5

for model_name, model_fn, model_type in models:

    print("\n" + "=" * 80)
    print(model_name)
    print("=" * 80)


    for feature_name, features in feature_sets.items():

        print(
            "\nFeature Set:",
            feature_name
        )


        for run in range(
            1,
            N_RUNS + 1
        ):


            # =================================================
            # CHECK WHETHER THIS RUN ALREADY EXISTS
            # =================================================

            run_identifier = (
                model_name,
                feature_name,
                run
            )

            if run_identifier in completed_runs:

                print(
                    f"SKIPPING existing run: "
                    f"{model_name} | "
                    f"{feature_name} | "
                    f"Run {run}"
                )

                continue


            print(
                "\n" +
                "-" * 80
            )

            print(
                f"{model_name} | "
                f"{feature_name} | "
                f"Run {run}/{N_RUNS}"
            )

            print(
                "-" * 80
            )


            # =================================================
            # SAFE NAMES
            # =================================================

            safe_model = make_safe_name(
                model_name
            )

            safe_feature = make_safe_name(
                feature_name
            )


            prefix = (
                f"{safe_model}_"
                f"{safe_feature}_"
                f"LB{LB}_"
                f"PL{PL}_"
                f"Run{run}"
            )


            # =================================================
            # REPRODUCIBILITY
            # =================================================

            run_seed = (
                SEED + run
            )

            np.random.seed(
                run_seed
            )

            random.seed(
                run_seed
            )

            tf.random.set_seed(
                run_seed
            )


            # =================================================
            # RAW DATA
            # =================================================

            X_raw = (
                df[features]
                .values
                .astype(np.float32)
            )

            y_raw = (
                df[[TARGET]]
                .values
                .astype(np.float32)
            )

            n_samples = len(df)


            # =================================================
            # CHRONOLOGICAL TRAIN / TEST SPLIT
            # =================================================

            raw_split = int(
                TRAIN_RATIO *
                n_samples
            )


            X_train_raw = (
                X_raw[:raw_split]
            )

            X_test_raw = (
                X_raw[raw_split:]
            )


            y_train_raw = (
                y_raw[:raw_split]
            )

            y_test_raw = (
                y_raw[raw_split:]
            )


            print(
                "Raw training samples:",
                len(X_train_raw)
            )

            print(
                "Raw testing samples:",
                len(X_test_raw)
            )


            # =================================================
            # FIT SCALERS ONLY ON TRAINING DATA
            # =================================================

            scaler_X = MinMaxScaler()

            scaler_y = MinMaxScaler()


            scaler_X.fit(
                X_train_raw
            )

            scaler_y.fit(
                y_train_raw
            )


            # =================================================
            # TRANSFORM DATA
            # =================================================

            X_train_scaled = (
                scaler_X.transform(
                    X_train_raw
                )
            )

            X_test_scaled = (
                scaler_X.transform(
                    X_test_raw
                )
            )


            y_train_scaled = (
                scaler_y.transform(
                    y_train_raw
                )
            )

            y_test_scaled = (
                scaler_y.transform(
                    y_test_raw
                )
            )


            # =================================================
            # CREATE TRAINING SEQUENCES
            # =================================================

            X_train_seq, y_train_seq = (
                create_multistep_sequences(
                    X_train_scaled,
                    y_train_scaled,
                    lookback=LB,
                    pred_length=PL
                )
            )


            # =================================================
            # CREATE TEST SEQUENCES
            # =================================================

            X_test_with_history = np.concatenate(
                [
                    X_train_scaled[-LB:],
                    X_test_scaled
                ],
                axis=0
            )


            y_test_with_history = np.concatenate(
                [
                    y_train_scaled[-LB:],
                    y_test_scaled
                ],
                axis=0
            )


            X_test_seq, y_test_seq = (
                create_multistep_sequences(
                    X_test_with_history,
                    y_test_with_history,
                    lookback=LB,
                    pred_length=PL
                )
            )


            # =================================================
            # REMOVE SEQUENCES EXTENDING BEYOND TEST PERIOD
            # =================================================

            expected_test_sequences = (
                len(y_test_raw)
                - PL
                + 1
            )


            X_test_seq = (
                X_test_seq[
                    :expected_test_sequences
                ]
            )

            y_test_seq = (
                y_test_seq[
                    :expected_test_sequences
                ]
            )


            print(
                "X_train:",
                X_train_seq.shape
            )

            print(
                "y_train:",
                y_train_seq.shape
            )

            print(
                "X_test:",
                X_test_seq.shape
            )

            print(
                "y_test:",
                y_test_seq.shape
            )


            # =================================================
            # MODEL TARGET
            # =================================================

            y_train_model = (
                y_train_seq.reshape(
                    y_train_seq.shape[0],
                    PL
                )
            )

            y_test_model = (
                y_test_seq.reshape(
                    y_test_seq.shape[0],
                    PL
                )
            )


            # =================================================
            # BUILD MODEL
            # =================================================

            model = model_fn(
                (
                    X_train_seq.shape[1],
                    X_train_seq.shape[2]
                )
            )


            # =================================================
            # CHECK OUTPUT
            # =================================================

            output_shape = (
                model.output_shape
            )


            if output_shape[-1] != PL:

                raise ValueError(
                    f"{model_name} output shape "
                    f"{output_shape} does not match "
                    f"PL={PL}"
                )


            print(
                "Model output:",
                output_shape
            )


            # =================================================
            # SAVE MODEL SUMMARY BEFORE TRAINING
            # =================================================

            summary_path = os.path.join(
                DIR_SUMMARY,
                prefix +
                "_ModelSummary.txt"
            )

            save_model_summary(
                model,
                summary_path
            )


            # =================================================
            # CALLBACKS
            # =================================================

            callbacks = []


            if model_type == "paper":

                callbacks.append(
                    LearningRateScheduler(
                        step_decay,
                        verbose=0
                    )
                )


            # =================================================
            # TRAINING
            # =================================================

            train_start = time.time()


            history = model.fit(

                X_train_seq,

                y_train_model,

                validation_split=0.10,

                epochs=EPOCHS,

                batch_size=BATCH_SIZE,

                callbacks=callbacks,

                verbose=0
            )


            training_time = (
                time.time()
                -
                train_start
            )


            print(
                f"Training time: "
                f"{training_time:.2f} sec"
            )


            # =================================================
            # PREDICTION
            # =================================================

            pred_start = time.time()


            y_pred_scaled = (
                model.predict(
                    X_test_seq,
                    verbose=0
                )
            )


            prediction_time = (
                time.time()
                -
                pred_start
            )


            # =================================================
            # INVERSE TRANSFORM
            # =================================================

            y_pred_inv = (
                scaler_y.inverse_transform(
                    y_pred_scaled
                )
            )


            y_true_inv = (
                scaler_y.inverse_transform(
                    y_test_model
                )
            )


            # =================================================
            # SAVE PREDICTIONS IMMEDIATELY
            # =================================================

            pred_data = {}


            for h in range(PL):

                pred_data[
                    f"Actual_t+{h+1}"
                ] = (
                    y_true_inv[:, h]
                )

                pred_data[
                    f"Predicted_t+{h+1}"
                ] = (
                    y_pred_inv[:, h]
                )

                pred_data[
                    f"Error_t+{h+1}"
                ] = (
                    y_true_inv[:, h]
                    -
                    y_pred_inv[:, h]
                )

                pred_data[
                    f"AbsoluteError_t+{h+1}"
                ] = np.abs(
                    y_true_inv[:, h]
                    -
                    y_pred_inv[:, h]
                )


            pred_df = pd.DataFrame(
                pred_data
            )


            prediction_path = os.path.join(
                DIR_PREDICTIONS,
                prefix +
                "_Predictions.csv"
            )


            pred_df.to_csv(
                prediction_path,
                index=False
            )


            print(
                "Predictions saved."
            )


            # =================================================
            # SAVE TRAINING HISTORY IMMEDIATELY
            # =================================================

            history_df = pd.DataFrame(
                history.history
            )


            history_path = os.path.join(
                DIR_HISTORY,
                prefix +
                "_History.csv"
            )


            history_df.to_csv(
                history_path,
                index=False
            )


            # =================================================
            # SAVE MODEL
            # =================================================

            model_path = os.path.join(
                DIR_MODELS,
                prefix +
                ".keras"
            )


            model.save(
                model_path
            )


            # =================================================
            # SAVE SCALERS
            # =================================================

            joblib.dump(
                scaler_X,
                os.path.join(
                    DIR_SCALERS,
                    prefix +
                    "_ScalerX.pkl"
                )
            )


            joblib.dump(
                scaler_y,
                os.path.join(
                    DIR_SCALERS,
                    prefix +
                    "_ScalerY.pkl"
                )
            )


            # =================================================
            # SAVE SEQUENCES
            # =================================================

            sequence_path = os.path.join(
                DIR_SEQUENCES,
                prefix +
                "_Sequences.npz"
            )


            np.savez_compressed(

                sequence_path,

                X_train_seq=X_train_seq,

                y_train_seq=y_train_seq,

                X_test_seq=X_test_seq,

                y_test_seq=y_test_seq,

                X_train_scaled=X_train_scaled,

                X_test_scaled=X_test_scaled,

                y_train_scaled=y_train_scaled,

                y_test_scaled=y_test_scaled
            )

            # =========================================================
            # SAVE TEST DATA + PREDICTIONS
            # =========================================================

            n_seq = len(y_pred_inv)
            aligned_index = pd.RangeIndex(start=0, stop=n_seq)
            test_export = pd.DataFrame(index=aligned_index)


            # -------------------------------------------------
            # ORIGINAL TEST FEATURES (SLICED TO n_seq)
            # -------------------------------------------------

            for i, feature in enumerate(features):

                test_export[
                    feature
                ] = X_test_raw[:n_seq, i]


            # -------------------------------------------------
            # PREDICTIONS
            # -------------------------------------------------

            for h in range(PL):

                test_export[
                    f"Actual_t+{h+1}"
                ] = y_true_inv[:, h]

                test_export[
                    f"Predicted_t+{h+1}"
                ] = y_pred_inv[:, h]

                test_export[
                    f"Error_t+{h+1}"
                ] = (
                    y_true_inv[:, h]
                    -
                    y_pred_inv[:, h]
                )

                test_export[
                    f"AbsoluteError_t+{h+1}"
                ] = np.abs(
                    y_true_inv[:, h]
                    -
                    y_pred_inv[:, h]
                )


            test_path = os.path.join(
                DIR_SEQUENCES,
                prefix +
                "_TestData_Predictions.csv"
            )


            test_export.to_csv(
                test_path,
                index=False
            )


            # =================================================
            # OVERALL METRICS
            # =================================================

            y_true_flat = (
                y_true_inv.flatten()
            )

            y_pred_flat = (
                y_pred_inv.flatten()
            )


            mse = mean_squared_error(
                y_true_flat,
                y_pred_flat
            )


            rmse = np.sqrt(
                mse
            )


            mae = mean_absolute_error(
                y_true_flat,
                y_pred_flat
            )


            r2 = r2_score(
                y_true_flat,
                y_pred_flat
            )


            mape = calculate_mape(
                y_true_flat,
                y_pred_flat
            )


            # =================================================
            # PER-HORIZON METRICS
            # =================================================

            current_horizon_results = []


            for h in range(PL):

                actual_h = (
                    y_true_inv[:, h]
                )

                pred_h = (
                    y_pred_inv[:, h]
                )


                mse_h = mean_squared_error(
                    actual_h,
                    pred_h
                )


                rmse_h = np.sqrt(
                    mse_h
                )


                mae_h = mean_absolute_error(
                    actual_h,
                    pred_h
                )


                r2_h = r2_score(
                    actual_h,
                    pred_h
                )


                mape_h = calculate_mape(
                    actual_h,
                    pred_h
                )


                horizon_row = {

                    "Model":
                        model_name,

                    "Feature Selection":
                        feature_name,

                    "Run":
                        run,

                    "Lookback":
                        LB,

                    "Prediction Length":
                        PL,

                    "Horizon":
                        h + 1,

                    "MSE":
                        mse_h,

                    "RMSE":
                        rmse_h,

                    "MAE":
                        mae_h,

                    "R2":
                        r2_h,

                    "MAPE":
                        mape_h
                }


                horizon_results.append(
                    horizon_row
                )

                current_horizon_results.append(
                    horizon_row
                )


            # =================================================
            # MAIN RESULT
            # =================================================

            result_row = {

                "Model":
                    model_name,

                "Feature Selection":
                    feature_name,

                "Selected Features":
                    ", ".join(
                        features
                    ),

                "Run":
                    run,

                "Lookback":
                    LB,

                "Prediction Length":
                    PL,

                "Train Ratio":
                    TRAIN_RATIO,

                "Test Ratio":
                    TEST_RATIO,

                "Raw Training Samples":
                    len(X_train_raw),

                "Raw Testing Samples":
                    len(X_test_raw),

                "Training Sequences":
                    len(X_train_seq),

                "Testing Sequences":
                    len(X_test_seq),

                "Number of Features":
                    len(features),

                "Epochs Used":
                    len(
                        history.history[
                            "loss"
                        ]
                    ),

                "MSE":
                    mse,

                "RMSE":
                    rmse,

                "MAE":
                    mae,

                "R2":
                    r2,

                "MAPE":
                    mape,

                "Training Time (sec)":
                    training_time,

                "Prediction Time (sec)":
                    prediction_time,

                "Run Seed":
                    run_seed
            }


            results.append(
                result_row
            )


            # =================================================
            # SAVE MAIN RESULTS IMMEDIATELY
            # =================================================

            pd.DataFrame(
                results
            ).to_csv(
                MAIN_RESULTS_FILE,
                index=False
            )


            # =================================================
            # SAVE HORIZON RESULTS IMMEDIATELY
            # =================================================

            pd.DataFrame(
                horizon_results
            ).to_csv(
                HORIZON_RESULTS_FILE,
                index=False
            )


            # =================================================
            # SAVE CONFIGURATION
            # =================================================

            config = {

                "Model":
                    model_name,

                "Model Type":
                    model_type,

                "Feature Selection":
                    feature_name,

                "Selected Features":
                    features,

                "Run":
                    run,

                "Lookback":
                    LB,

                "Prediction Length":
                    PL,

                "Batch Size":
                    BATCH_SIZE,

                "Maximum Epochs":
                    EPOCHS,

                "Actual Epochs Used":
                    len(
                        history.history[
                            "loss"
                        ]
                    ),

                "Train Ratio":
                    TRAIN_RATIO,

                "Test Ratio":
                    TEST_RATIO,

                "Target":
                    TARGET,

                "Total Raw Samples":
                    len(df),

                "Raw Training Samples":
                    len(X_train_raw),

                "Raw Testing Samples":
                    len(X_test_raw),

                "Training Sequences":
                    len(X_train_seq),

                "Testing Sequences":
                    len(X_test_seq),

                "Number of Features":
                    len(features),

                "Feature Names":
                    features,

                "Input Sequence Shape":
                    list(
                        X_train_seq.shape
                    ),

                "Target Sequence Shape":
                    list(
                        y_train_seq.shape
                    ),

                "Model Output Shape":
                    list(
                        model.output_shape
                    ),

                "MSE":
                    mse,

                "RMSE":
                    rmse,

                "MAE":
                    mae,

                "R2":
                    r2,

                "MAPE":
                    mape,

                "Training Time (sec)":
                    training_time,

                "Prediction Time (sec)":
                    prediction_time,

                "Global Seed":
                    SEED,

                "Run Seed":
                    run_seed
            }


            config_path = os.path.join(
                DIR_CONFIG,
                prefix +
                "_Config.json"
            )


            with open(
                config_path,
                "w"
            ) as f:

                json.dump(
                    config,
                    f,
                    indent=4,
                    default=str
                )


            # =================================================
            # SAVE TRAINING GRAPH
            # =================================================

            save_training_graph(
                history,
                prefix
            )


            # =================================================
            # SAVE PREDICTION GRAPHS
            # =================================================

            save_prediction_graphs(
                y_true_inv,
                y_pred_inv,
                prefix
            )


            # =================================================
            # STORE IN MEMORY
            # =================================================

            predictions[
                (
                    model_name,
                    feature_name,
                    run
                )
            ] = (
                y_true_inv,
                y_pred_inv
            )


            histories[
                (
                    model_name,
                    feature_name,
                    run
                )
            ] = history


            # =================================================
            # MARK AS COMPLETED
            # =================================================

            completed_runs.add(
                run_identifier
            )


            # =================================================
            # PRINT RESULTS
            # =================================================

            print(
                "\nRUN COMPLETED"
            )

            print(
                f"MSE   = {mse:.6f}"
            )

            print(
                f"RMSE  = {rmse:.6f}"
            )

            print(
                f"MAE   = {mae:.6f}"
            )

            print(
                f"R²    = {r2:.6f}"
            )

            print(
                f"MAPE  = {mape:.4f}%"
            )

            print(
                f"Training Time = "
                f"{training_time:.2f}s"
            )

            print(
                f"Prediction Time = "
                f"{prediction_time:.4f}s"
            )

            print(
                "All run data saved successfully."
            )

Previously completed runs: 0

Model 7: pCNN_BiLSTM

Feature Set: Correlation

--------------------------------------------------------------------------------
Model 7: pCNN_BiLSTM | Correlation | Run 1/5
--------------------------------------------------------------------------------
Raw training samples: 5610
Raw testing samples: 3022
X_train: (5575, 24, 5)
y_train: (5575, 12, 1)
X_test: (3011, 24, 5)
y_test: (3011, 12, 1)
Model output: (None, 12)


Training time: 75.59 sec
Predictions saved.

RUN COMPLETED
MSE   = 9998.035156
RMSE  = 99.990175
MAE   = 16.630230
R²    = -0.002773
MAPE  = 16.9732%
Training Time = 75.59s
Prediction Time = 1.0024s
All run data saved successfully.

--------------------------------------------------------------------------------
Model 7: pCNN_BiLSTM | Correlation | Run 2/5
--------------------------------------------------------------------------------
Raw training samples: 5610
Raw testing samples: 3022
X_train: (5575, 24, 5)
y_train: (5575, 12, 1)
X_test: (3011, 24, 5)
y_test: (3011, 12, 1)
Model output: (None, 12)


Training time: 68.43 sec
Predictions saved.

RUN COMPLETED
MSE   = 10052.966797
RMSE  = 100.264484
MAE   = 14.792898
R²    = -0.008282
MAPE  = 13.2774%
Training Time = 68.43s
Prediction Time = 0.8828s
All run data saved successfully.

--------------------------------------------------------------------------------
Model 7: pCNN_BiLSTM | Correlation | Run 3/5
--------------------------------------------------------------------------------
Raw training samples: 5610
Raw testing samples: 3022
X_train: (5575, 24, 5)
y_train: (5575, 12, 1)
X_test: (3011, 24, 5)
y_test: (3011, 12, 1)
Model output: (None, 12)


Training time: 68.43 sec
Predictions saved.

RUN COMPLETED
MSE   = 10046.528320
RMSE  = 100.232372
MAE   = 14.130943
R²    = -0.007637
MAPE  = 12.1186%
Training Time = 68.43s
Prediction Time = 0.8860s
All run data saved successfully.

--------------------------------------------------------------------------------
Model 7: pCNN_BiLSTM | Correlation | Run 4/5
--------------------------------------------------------------------------------
Raw training samples: 5610
Raw testing samples: 3022
X_train: (5575, 24, 5)
y_train: (5575, 12, 1)
X_test: (3011, 24, 5)
y_test: (3011, 12, 1)
Model output: (None, 12)


Training time: 68.96 sec
Predictions saved.

RUN COMPLETED
MSE   = 10052.902344
RMSE  = 100.264163
MAE   = 14.487668
R²    = -0.008276
MAPE  = 12.7242%
Training Time = 68.96s
Prediction Time = 0.8889s
All run data saved successfully.

--------------------------------------------------------------------------------
Model 7: pCNN_BiLSTM | Correlation | Run 5/5
--------------------------------------------------------------------------------
Raw training samples: 5610
Raw testing samples: 3022
X_train: (5575, 24, 5)
y_train: (5575, 12, 1)
X_test: (3011, 24, 5)
y_test: (3011, 12, 1)
Model output: (None, 12)


...

Training time: 68.69 sec
Predictions saved.

RUN COMPLETED
MSE   = 9999.904297
RMSE  = 99.999521
MAE   = 16.874573
R²    = -0.002960
MAPE  = 17.4373%
Training Time = 68.69s
Prediction Time = 0.8741s
All run data saved successfully.

--------------------------------------------------------------------------------
Model 7: pCNN_BiLSTM | SelectKBest | Run 2/5
--------------------------------------------------------------------------------
Raw training samples: 5610
Raw testing samples: 3022
X_train: (5575, 24, 5)
y_train: (5575, 12, 1)
X_test: (3011, 24, 5)
y_test: (3011, 12, 1)
Model output: (None, 12)


Training time: 68.56 sec
Predictions saved.

RUN COMPLETED
MSE   = 10052.224609
RMSE  = 100.260783
MAE   = 14.688711
R²    = -0.008208
MAPE  = 13.1049%
Training Time = 68.56s
Prediction Time = 0.8871s
All run data saved successfully.

--------------------------------------------------------------------------------
Model 7: pCNN_BiLSTM | SelectKBest | Run 3/5
--------------------------------------------------------------------------------
Raw training samples: 5610
Raw testing samples: 3022
X_train: (5575, 24, 5)
y_train: (5575, 12, 1)
X_test: (3011, 24, 5)
y_test: (3011, 12, 1)
Model output: (None, 12)


Training time: 70.59 sec
Predictions saved.

RUN COMPLETED
MSE   = 10035.460938
RMSE  = 100.177148
MAE   = 15.180145
R²    = -0.006527
MAPE  = 14.0773%
Training Time = 70.59s
Prediction Time = 0.9007s
All run data saved successfully.

--------------------------------------------------------------------------------
Model 7: pCNN_BiLSTM | SelectKBest | Run 4/5
--------------------------------------------------------------------------------
Raw training samples: 5610
Raw testing samples: 3022
X_train: (5575, 24, 5)
y_train: (5575, 12, 1)
X_test: (3011, 24, 5)
y_test: (3011, 12, 1)
Model output: (None, 12)


Training time: 68.81 sec
Predictions saved.

RUN COMPLETED
MSE   = 10016.870117
RMSE  = 100.084315
MAE   = 15.321553
R²    = -0.004662
MAPE  = 14.4550%
Training Time = 68.81s
Prediction Time = 0.8756s
All run data saved successfully.

--------------------------------------------------------------------------------
Model 7: pCNN_BiLSTM | SelectKBest | Run 5/5
--------------------------------------------------------------------------------
Raw training samples: 5610
Raw testing samples: 3022
X_train: (5575, 24, 5)
y_train: (5575, 12, 1)
X_test: (3011, 24, 5)
y_test: (3011, 12, 1)
Model output: (None, 12)


Training time: 68.62 sec
Predictions saved.

RUN COMPLETED
MSE   = 10033.861328
RMSE  = 100.169164
MAE   = 14.710458
R²    = -0.006366
MAPE  = 13.2598%
Training Time = 68.62s
Prediction Time = 0.9224s
All run data saved successfully.




***First, reconnect to my saved results:***